# SMS Text Classification — Ham or Spam?

This notebook trains an **LSTM neural network** to classify SMS messages as **ham** (a normal message) or **spam** (an ad or scam).


In [15]:
# --- Imports & reproducibility ---
import os
import re
import string
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Set seeds so results are reproducible run-to-run
SEED = 10
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow", tf.__version__)


TensorFlow 2.20.0


## 1. Download & load the data

The **SMS Spam Collection** dataset comes from freeCodeCamp. It has two TSV files:

- `train-data.tsv` — messages used to **train** the model
- `valid-data.tsv` — messages used to **test** the model

Each row has two columns: `type` (`ham` or `spam`) and `message` (the text).


In [16]:
# Download the data files (skip if already present)
if not os.path.exists("train-data.tsv"):
    !wget -q https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
if not os.path.exists("valid-data.tsv"):
    !wget -q https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"


In [17]:
train = pd.read_csv(
    train_file_path,
    encoding="ISO-8859-1",
    sep="\t",
    header=0,
    names=["type", "message"],
    usecols=["type", "message"],
    dtype={"type": "str", "message": "str"},
)

test = pd.read_csv(
    test_file_path,
    encoding="ISO-8859-1",
    sep="\t",
    header=0,
    names=["type", "message"],
    usecols=["type", "message"],
    dtype={"type": "str", "message": "str"},
)

print("Training messages:", len(train))
print("Test messages:    ", len(test))


Training messages: 4178
Test messages:     1391


In [18]:
# Convert text labels to numbers: ham -> 0, spam -> 1
label_map = {"ham": 0, "spam": 1}

train["type"] = train["type"].map(label_map).astype(int)
test["type"] = test["type"].map(label_map).astype(int)


## 2. Turn words into numbers

Neural networks can't read text — they only understand numbers. So we:

1. **Clean** each message (lowercase, strip HTML tags, remove punctuation)
2. **Tokenize** — split into individual words
3. **Vectorize** — map each word to a unique integer ID

The `TextVectorization` layer handles all three steps. We `adapt()` it on the training messages to build the vocabulary.


In [19]:
# Custom text cleaning: lowercase, strip <br /> tags, remove punctuation
def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, "<br />", " ")
    return tf.strings.regex_replace(
        stripped_html, "[%s]" % re.escape(string.punctuation), ""
    )


vocab_size = 10000  # maximum number of unique words to keep
sequence_length = 150  # pad/truncate every message to 150 tokens

vectorize_layer = keras.layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

# Build the vocabulary from the training messages
train_array = np.array(train["message"])
vectorize_layer.adapt(train_array)


## 3. Build the neural network

Architecture (left to right = data flow):

| Layer                  | What it does                                                   |
| ---------------------- | -------------------------------------------------------------- |
| **TextVectorization**  | Converts raw text → integer sequences                          |
| **Embedding**          | Converts each integer ID → a 64-dim vector of learned meaning  |
| **Bidirectional LSTM** | Reads the sequence forward _and_ backward, remembering context |
| **Dense (ReLU)**       | A standard hidden layer that combines features                 |
| **Dropout**            | Randomly disables 30% of neurons to prevent overfitting        |
| **Dense (sigmoid)**    | Outputs a single number between 0 (ham) and 1 (spam)           |


In [20]:
model = keras.Sequential(
    [
        vectorize_layer,
        keras.layers.Embedding(vocab_size, 64, name="embedding"),
        keras.layers.Bidirectional(keras.layers.LSTM(64)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"),
    ]
)

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 4. Train the model

Two important details:

- **Class weights** — the dataset has far more `ham` than `spam` messages. Without weights the model would just guess "ham" for everything. Class weights tell the model to pay extra attention to spam, so it learns both types fairly.
- **Shuffling** — we shuffle the data before the 80/20 validation split so the validation set is representative.

Early stopping halts training when validation accuracy stops improving.


In [21]:
# Compute class weights to counteract the ham/spam imbalance
class_counts = np.bincount(train["type"])
total = len(train)
class_weight = {
    i: total / (len(class_counts) * class_counts[i]) for i in range(len(class_counts))
}
print("Class counts:", dict(enumerate(class_counts)))
print("Class weights:", class_weight)

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    verbose=1,
    mode="max",
    restore_best_weights=True,
)

# Shuffle before validation_split so the validation set is representative
train_shuffled = train.sample(frac=1, random_state=SEED).reset_index(drop=True)

history = model.fit(
    tf.convert_to_tensor(train_shuffled["message"].to_numpy()),
    tf.convert_to_tensor(train_shuffled["type"].to_numpy()),
    callbacks=[early_stopping],
    epochs=40,
    validation_split=0.2,
    class_weight=class_weight,
)


Class counts: {0: np.int64(3618), 1: np.int64(560)}
Class weights: {0: np.float64(0.5773908236594804), 1: np.float64(3.7303571428571427)}
Epoch 1/40
105/105 ━━━━━━━━━━━━━━━━━━━━ 22s 168ms/step - accuracy: 0.8665 - loss: 0.3452 - val_accuracy: 0.8732 - val_loss: 0.3270
Epoch 2/40
105/105 ━━━━━━━━━━━━━━━━━━━━ 19s 177ms/step - accuracy: 0.9835 - loss: 0.0843 - val_accuracy: 0.9821 - val_loss: 0.0694
Epoch 3/40
105/105 ━━━━━━━━━━━━━━━━━━━━ 18s 158ms/step - accuracy: 0.9952 - loss: 0.0280 - val_accuracy: 0.9797 - val_loss: 0.0673
Epoch 4/40
105/105 ━━━━━━━━━━━━━━━━━━━━ 17s 158ms/step - accuracy: 0.9970 - loss: 0.0219 - val_accuracy: 0.9617 - val_loss: 0.1261
Epoch 5/40
105/105 ━━━━━━━━━━━━━━━━━━━━ 21s 161ms/step - accuracy: 0.9979 - loss: 0.0098 - val_accuracy: 0.9833 - val_loss: 0.0653
Epoch 6/40
105/105 ━━━━━━━━━━━━━━━━━━━━ 17s 163ms/step - accuracy: 0.9997 - loss: 0.0029 - val_accuracy: 0.9833 - val_loss: 0.0771
Epoch 7/40
105/105 ━━━━━━━━━━━━━━━━━━━━ 20s 163ms/step - accuracy: 1.0000 - 

In [22]:
loss, accuracy = model.evaluate(
    tf.convert_to_tensor(test["message"].to_numpy()),
    tf.convert_to_tensor(test["type"].to_numpy()),
)

print("Loss:    ", loss)
print("Accuracy:", accuracy)


44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.9878 - loss: 0.0648
Loss:     0.06476878374814987
Accuracy: 0.9877786040306091


## 5. The `predict_message()` function

Give it a text message → it returns `[probability, label]`:

- `probability` is a float between 0 (ham) and 1 (spam)
- `label` is `'ham'` or `'spam'` (threshold = 0.5)


In [23]:
def predict_message(pred_text):
    pred = model.predict(tf.convert_to_tensor([pred_text]), verbose=0)[0]
    return [float(pred[0]), "spam" if pred[0] >= 0.5 else "ham"]


# Quick demo
demo_messages = [
    "how are you doing today",
    "sale today! to stop texts call 98912460324",
    "i dont want to go. can we try it a different day? available sat",
    "our new mobile video service is live. just install on your phone to start watching.",
    "you have won £1000 cash! call to claim your prize.",
    "i'll bring it tomorrow. don't forget the milk.",
    "wow, is your arm alright. that happened to me one time too",
]

for msg in demo_messages:
    print(predict_message(msg))


[6.409892375813797e-05, 'ham']
[0.9997788667678833, 'spam']
[5.199292900215369e-06, 'ham']
[0.9998467564582825, 'spam']
[0.9999985098838806, 'spam']
[1.634297404962126e-05, 'ham']
[5.762389628216624e-05, 'ham']


In [24]:
# This cell is used to test the function and model.
def test_predictions():
    test_messages = [
        "how are you doing today",
        "sale today! to stop texts call 98912460324",
        "i dont want to go. can we try it a different day? available sat",
        "our new mobile video service is live. just install on your phone to start watching.",
        "you have won £1000 cash! call to claim your prize.",
        "i'll bring it tomorrow. don't forget the milk.",
        "wow, is your arm alright. that happened to me one time too",
    ]

    test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
    passed = True

    for msg, ans in zip(test_messages, test_answers):
        prediction = predict_message(msg)
        if prediction[1] != ans:
            passed = False

    if passed:
        print("You passed the challenge. Great job!")
    else:
        print("You haven't passed yet. Keep trying.")


test_predictions()


You passed the challenge. Great job!
